---
## 1. Importation bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
import matplotlib.dates as mdates
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.ensemble import RandomForestClassifier as rf
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.svm import SVR
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV
import scipy.interpolate as spi
from scipy.interpolate import griddata
import h3
from imblearn.under_sampling import RandomUnderSampler
import xgboost as xgb
from collections import Counter
from scipy.stats import randint
from sklearn.dummy import DummyClassifier
import optuna

# dataset créé dans Projet_IA_Stage_Dep13.ipynb
merged_data13 = pd.read_csv('C:/Users/wittl/OneDrive/Documents/Cours/Université et Etudes/IUT Belfort/S6/Stage/results_img_csv/merged_data13.csv')
# dataset créé dans Projet_IA_Stage_Dep34.ipynb
merged_data34 = pd.read_csv('C:/Users/wittl/OneDrive/Documents/Cours/Université et Etudes/IUT Belfort/S6/Stage/results_img_csv/merged_data34.csv')

In [ ]:
import os

output_path = "../results_img_csv"
os.makedirs(output_path, exist_ok=True)

---
Verif données

In [ ]:
merged_data13.describe()

In [ ]:
merged_data34.describe()

---
## 2. Comparaison

---
#### Température

In [ ]:
"""merged_data13['date'] = pd.to_datetime(merged_data13['date'])
merged_data13['annee'] = merged_data13['date'].dt.year

plt.plot(merged_data13['date'], merged_data13['temp'], label='Température')
plt.xlabel("Créneau horaire")
plt.ylabel("Température (°C)")
plt.title("Moyenne températures des Bouches-du-Rhône")
plt.show()

merged_data34['date'] = pd.to_datetime(merged_data34['date'])
merged_data34['annee'] = merged_data34['date'].dt.year

plt.plot(merged_data34['date'], merged_data34['temp'], label='Température')
plt.xlabel("Créneau horaire")
plt.ylabel("Température (°C)")
plt.title("Moyenne températures de l'Hérault")
plt.show()"""

---
#### Humidité

In [ ]:
"""merged_data13['date'] = pd.to_datetime(merged_data13['date'])
merged_data13['annee'] = merged_data13['date'].dt.year

plt.plot(merged_data13['date'], merged_data13['rhum'], label='Humidité')
plt.xlabel("Créneau horaire")
plt.ylabel("Humidité")
plt.title("Moyenne Humidité des Bouches-du-Rhône")
plt.show()

merged_data34['date'] = pd.to_datetime(merged_data34['date'])
merged_data34['annee'] = merged_data34['date'].dt.year

plt.plot(merged_data34['date'], merged_data34['rhum'], label='Humidité')
plt.xlabel("Créneau horaire")
plt.ylabel("Humidité")
plt.title("Moyenne Humidité de l'Hérault")
plt.show()"""

---
#### Vitesse vent

In [ ]:
"""merged_data13['date'] = pd.to_datetime(merged_data13['date'])
merged_data13['annee'] = merged_data13['date'].dt.year

plt.plot(merged_data13['date'], merged_data13['wspd'], label='Vitesse du vent (km/h)')
plt.xlabel("Créneau horaire")
plt.ylabel("Vitesse du vent (km/h)")
plt.title("Moyenne vitesse vent des Bouches-du-Rhône")
plt.show()

merged_data34['date'] = pd.to_datetime(merged_data34['date'])
merged_data34['annee'] = merged_data34['date'].dt.year

plt.plot(merged_data34['date'], merged_data34['wspd'], label='Vitesse du vent (km/h)')
plt.xlabel("Créneau horaire")
plt.ylabel("Vitesse du vent (km/h)")
plt.title("Moyenne vitesse vent de l'Hérault")
plt.show()"""

---
#### Précipitations

In [ ]:
"""merged_data13['date'] = pd.to_datetime(merged_data13['date'])
merged_data13['annee'] = merged_data13['date'].dt.year

plt.plot(merged_data13['date'], merged_data13['prcp'], label='Précipitations')
plt.xlabel("Créneau horaire")
plt.ylabel("Précipitations")
plt.title("Moyenne vitesse vent des Bouches-du-Rhône")
plt.ylim(0, 300)
plt.show()

merged_data34['date'] = pd.to_datetime(merged_data34['date'])
merged_data34['annee'] = merged_data34['date'].dt.year

plt.plot(merged_data34['date'], merged_data34['prcp'], label='Précipitations')
plt.xlabel("Créneau horaire")
plt.ylabel("Précipitations")
plt.title("Moyenne vitesse vent de l'Hérault")
plt.ylim(0, 300)
plt.show()"""

---
## 3. Modélisation avec les 2 datasets

---
#### 3.1 Preprocessing

---
Encodage datasets + création feature pour différencier le département + concaténation datasets pour en faire un seul

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

merged_data13["departement"] = 13
merged_data34["departement"] = 34

merged_data = pd.concat([merged_data13, merged_data34], ignore_index=True)
cat_columns = merged_data.select_dtypes(include=['object']).columns

label_encoders = {}
for col in cat_columns:
    le = LabelEncoder()
    merged_data[col] = le.fit_transform(merged_data[col])
    label_encoders[col] = le

merged_data.describe()

#merged_data.to_csv('../results_img_csv/merged_data.csv', index=False)

In [ ]:
features = merged_data.drop(columns=['date', 'incendie_present']).columns.tolist()
print("Features utilisées :", features)
target = 'incendie_present'

train_data = merged_data[merged_data['date'] <= '2022-12-31']
test_data = merged_data[merged_data['date'] >= '2023-01-01']

X_train = train_data[features]
y_train = train_data[target]
X_test = test_data[features]
y_test = test_data[target]

imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

smote = SMOTE(sampling_strategy='auto', random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print(f"Données d'entraînement : {X_train_resampled.shape[0]} lignes")
print(f"Données de test : {X_test_scaled.shape[0]} lignes")

---
#### 3.2 Entrainement/test modèles

In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=5),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    'Extra Trees': ExtraTreesClassifier(n_estimators=100, max_depth=5, random_state=42),
    'CatBoost': CatBoostClassifier(learning_rate=0.1, depth=5, iterations=100, verbose=0, random_state=42),
    'LightGBM': LGBMClassifier(max_depth=5, n_estimators=100, verbose=-1, random_state=42),
    'XGBoost': XGBClassifier(max_depth=5, n_estimators=100, learning_rate=0.1, random_state=42)
}

"""models = {
    'Decision Tree': DecisionTreeClassifier(
        max_depth=5,
        min_samples_split=10,
        min_samples_leaf=5,
        criterion='entropy',
        random_state=42),

    'Random Forest': RandomForestClassifier(
        n_estimators=160,
        max_depth=7,
        min_samples_split=10,
        min_samples_leaf=3,
        max_features='sqrt',
        bootstrap=True,
        random_state=42),

    'Extra Trees': ExtraTreesClassifier(
        n_estimators=1250,
        max_depth=20,
        min_samples_split=12,
        min_samples_leaf=1,
        max_features='sqrt',
        bootstrap=False,
        random_state=42),

    'CatBoost': CatBoostClassifier(
        learning_rate=0.01,
        depth=5,
        iterations=500,
        l2_leaf_reg=3,
        border_count=64,
        random_strength=2,
        verbose=0,
        random_state=42),

    'LightGBM': LGBMClassifier(
        max_depth=3,
        n_estimators=400,
        learning_rate=0.07,
        num_leaves=31,
        min_child_samples=10,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=42,
        verbose=-1),

    'XGBoost': XGBClassifier(
        enable_categorical=False,
        eval_metric='mlogloss',
        max_depth=5,
        n_estimators=500,
        learning_rate=0.009,
        subsample=0.7,
        colsample_bytree=0.8,
        gamma=0.01,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=42)
}"""

results = {}

In [ ]:
for model_name, model in models.items():
    print(f"\nEntraînement du modèle : {model_name}")
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    accuracy = accuracy_score(y_test, y_pred)
    results[model_name] = accuracy
    print(f"Score du modèle {model_name}: {accuracy:.5f}")

In [ ]:
print("Distribution y_train :")
print(y_train.value_counts(normalize=True))
print("Distribution y_test :")
print(y_test.value_counts(normalize=True))

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    print(f"\n=== Résultats pour {name} ===")
    y_pred = model.predict(X_test)
    y_proba = None
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f"Accuracy : {accuracy:.6f}")
    print(f"Precision : {precision:.6f}")
    print(f"Recall : {recall:.6f}")
    print(f"F1-Score : {f1:.6f}")

    return accuracy, precision, recall, f1

In [ ]:
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    evaluate_model(name, model, X_train_resampled, y_train_resampled, X_test_scaled, y_test)

---
#### 3.3 Amélioration / optimisation

In [ ]:
from sklearn.model_selection import cross_val_score
import numpy as np

results = {}

for model_name, model in models.items():
    try:
        cv_scores = cross_val_score(model, X_train_resampled, y_train_resampled, cv=5, scoring='f1')
        results[model_name] = {
            'scores': cv_scores, 'mean_score': np.mean(cv_scores), 'std_dev': np.std(cv_scores)
        }
    except Exception as e:
        print(f"Erreur avec le modèle {model_name}: {e}")
        results[model_name] = {
            'scores': None, 'mean_score': None, 'std_dev': None
        }
for model_name, result in results.items():
    print(f"Modèle: {model_name}")
    if result['scores'] is not None:
        print(f"Scores de cross-validation: {result['scores']}")
        print(f"Moyenne des scores: {result['mean_score']}")
        print(f"Écart-type des scores: {result['std_dev']}")
    else:
        print("Erreur avec ce modèle")
    print('-' * 30)

In [ ]:
# Pour Cross Validation
scoring_metrics = ['accuracy', 'precision', 'recall', 'f1']

cv_results = {}

for model_name, model in models.items():
    try:
        scores = cross_validate(model, X_train_resampled, y_train_resampled, cv=5, scoring=scoring_metrics)
        cv_results[model_name] = {
            'accuracy': np.mean(scores['test_accuracy']),
            'precision': np.mean(scores['test_precision']),
            'recall': np.mean(scores['test_recall']),
            'f1-score': np.mean(scores['test_f1']),
            'accuracy_std': np.std(scores['test_accuracy']),
            'precision_std': np.std(scores['test_precision']),
            'recall_std': np.std(scores['test_recall']),
            'f1-score_std': np.std(scores['test_f1']),
        }

    except Exception as e:
        print(f"Erreur avec le modèle {model_name}: {e}")
        cv_results[model_name] = None

for model_name, result in cv_results.items():
    print(f"\n🔹 Modèle: {model_name}")
    if result:
        print(f"Accuracy: {result['accuracy']:.4f} ± {result['accuracy_std']:.4f}")
        print(f"Precision: {result['precision']:.4f} ± {result['precision_std']:.4f}")
        print(f"Recall: {result['recall']:.4f} ± {result['recall_std']:.4f}")
        print(f"F1-Score: {result['f1-score']:.4f} ± {result['f1-score_std']:.4f}")
    else:
        print("⚠️ Erreur avec ce modèle")
    print('-' * 30)

---
Graphique visualisation métriques

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

metrics_results = {}

for name, model in models.items():
    accuracy, precision, recall, f1 = evaluate_model(name, model, X_train_resampled, y_train_resampled, X_test_scaled, y_test)

    metrics_results[name] = {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    }

model_names = list(metrics_results.keys())
accuracy = [metrics['Accuracy'] for metrics in metrics_results.values()]
precision = [metrics['Precision'] for metrics in metrics_results.values()]
recall = [metrics['Recall'] for metrics in metrics_results.values()]
f1_score = [metrics['F1-Score'] for metrics in metrics_results.values()]

average_metrics = {
    'Accuracy': sum([metrics['Accuracy'] for metrics in metrics_results.values()]) / len(metrics_results),
    'Precision': sum([metrics['Precision'] for metrics in metrics_results.values()]) / len(metrics_results),
    'Recall': sum([metrics['Recall'] for metrics in metrics_results.values()]) / len(metrics_results),
    'F1-Score': sum([metrics['F1-Score'] for metrics in metrics_results.values()]) / len(metrics_results)
}

print("\nMoyenne des métriques :")
for metric, value in average_metrics.items():
    print(f"{metric} : {value:.6f}")
width = 0.2
x = np.arange(len(model_names))

fig, ax = plt.subplots(figsize=(10, 6))

ax.bar(x - 1.5 * width, accuracy, width, label='Accuracy')
ax.bar(x - 0.5 * width, precision, width, label='Precision')
ax.bar(x + 0.5 * width, recall, width, label='Recall')
ax.bar(x + 1.5 * width, f1_score, width, label='F1-Score')

ax.set_xlabel('Modèle')
ax.set_ylabel('Valeurs des métriques')
ax.set_title('Comparaison des performances des modèles')
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.legend()

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---
#### 3.4 Tests autres techniques

---
Application 3e ensemble de validation

In [ ]:
# ===== 3e ensemble de validation avec séparation en fonction de la date =====

train_val_data = merged_data[merged_data['date'] <= '2022-12-31']
test_data = merged_data[merged_data['date'] >= '2022-12-31']

X_train_val_3 = train_val_data.drop(columns=['incendie_present'])
y_train_val_3 = train_val_data['incendie_present']

X_test_3 = test_data.drop(columns=['incendie_present'])
y_test_3 = test_data['incendie_present']

X_train_3, X_val_3, y_train_3, y_val_3 = train_test_split(X_train_val_3, y_train_val_3, test_size=0.2, random_state=42)

print(f"Taille des ensembles :\n- Train: {X_train_3.shape}\n- Validation: {X_val_3.shape}\n- Test: {X_test_3.shape}")

X_train_no_date = X_train_3.drop(columns=['date'])
X_val_no_date = X_val_3.drop(columns=['date'])
X_test_no_date = X_test_3.drop(columns=['date'])

scaler = StandardScaler()

X_train_scaled_3 = scaler.fit_transform(X_train_no_date)
X_val_scaled_3 = scaler.transform(X_val_no_date)
X_test_scaled_3 = scaler.transform(X_test_no_date)

# ===== 3e ensemble de validation avec séparation aléatoire (lignes prises aléatoirement) =====

"""merged_data = merged_data.drop_duplicates()

X = merged_data.drop(columns=['incendie_present'])
y = merged_data['incendie_present']

print(f"Dimensions de X après suppression des doublons: {X.shape}")
print(f"Dimensions de y après suppression des doublons: {y.shape}")

X = X.reset_index(drop=True)
y = y.reset_index(drop=True)
assert X.shape[0] == y.shape[0], "Les dimensions de X et y ne correspondent toujours pas."

X_train_val_3, X_test_3, y_train_val_3, y_test_3 = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_3, X_val_3, y_train_3, y_val_3 = train_test_split(X_train_val_3, y_train_val_3, test_size=0.2, random_state=42)

print(f"Taille des ensembles :\n- Train: {X_train_3.shape}\n- Validation: {X_val_3.shape}\n- Test: {X_test_3.shape}")

X_train_no_date = X_train_3.drop(columns=['date'])
X_val_no_date = X_val_3.drop(columns=['date'])
X_test_no_date = X_test_3.drop(columns=['date'])

scaler = StandardScaler()

X_train_scaled_3 = scaler.fit_transform(X_train_no_date)
X_val_scaled_3 = scaler.transform(X_val_no_date)
X_test_scaled_3 = scaler.transform(X_test_no_date)"""

---
Modèle de persistance

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

merged_data = merged_data.sort_values('date')

merged_data['prev_incendie_present'] = merged_data['incendie_present'].shift(1)

merged_data['prev_incendie_present'].fillna(0, inplace=True)

y_val_pred = merged_data.loc[X_val_3.index, 'prev_incendie_present']
val_accuracy = accuracy_score(y_val_3, y_val_pred)
val_precision = precision_score(y_val_3, y_val_pred)
val_recall = recall_score(y_val_3, y_val_pred)
val_f1 = f1_score(y_val_3, y_val_pred)

print(f"\n🔹 **Performance du modèle de persistance sur l'ensemble de Validation :**")
print(f"✅ Accuracy: {val_accuracy:.4f}")
print(f"✅ Precision: {val_precision:.4f}")
print(f"✅ Recall: {val_recall:.4f}")
print(f"✅ F1-Score: {val_f1:.4f}")

y_test_pred = merged_data.loc[X_test_3.index, 'prev_incendie_present']
test_accuracy = accuracy_score(y_test_3, y_test_pred)
test_precision = precision_score(y_test_3, y_test_pred)
test_recall = recall_score(y_test_3, y_test_pred)
test_f1 = f1_score(y_test_3, y_test_pred)

print(f"\n🔹 **Performance du modèle de persistance sur l'ensemble de Test :**")
print(f"✅ Accuracy: {test_accuracy:.4f}")
print(f"✅ Precision: {test_precision:.4f}")
print(f"✅ Recall: {test_recall:.4f}")
print(f"✅ F1-Score: {test_f1:.4f}")

---
Recherche paramètres optimaux XGBoost

In [ ]:
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, f1_score

xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    tree_method='hist',
    random_state=42
)

param_distributions = {
    'learning_rate': [0.005, 0.01, 0.05, 0.1, 0.2, 0.3, 0.5],
    'max_depth': [3, 4, 5, 6, 7, 8],
    'subsample': [0.2, 0.4, 0.6, 0.8, 1.0],
    'colsample_bytree': [0.2, 0.4, 0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 5, 7, 10],
    'gamma': [0.001, 0.01, 0.1, 0.3, 0.5, 1.0],
    'scale_pos_weight': [1.0, 1.5, 2.0, 3.0, 5.0],
    'lambda': [0.001, 0.01, 0.1, 0.5, 1.0],
    'alpha': [0.001, 0.01, 0.1, 0.5, 1.0],
    'max_delta_step': [0, 1, 3, 5, 7, 10]
}

random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_distributions,
    n_iter=10,
    scoring=make_scorer(f1_score),
    cv=5,
    verbose=1,
    n_jobs=-1
)

random_search.fit(X_train_scaled_3, y_train_3)

best_params = random_search.best_params_
print("\n✅ Meilleurs hyperparamètres trouvés :", best_params)

---
Optimisation XGBoost

In [ ]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

dtrain = xgb.DMatrix(X_train_scaled_3, label=y_train_3)
dval = xgb.DMatrix(X_val_scaled_3, label=y_val_3)
dtest = xgb.DMatrix(X_test_scaled_3, label=y_test_3)

params = {
    'objective': 'binary:logistic', 'eval_metric': 'logloss',
    'learning_rate': 0.01, 'max_depth': 7,
    'subsample': 0.6, 'colsample_bytree': 0.8,
    'min_child_weight': 5, 'gamma': 0.01,
    'scale_pos_weight': 3.0, 'lambda': 0.01,
    'alpha': 0.01, 'max_delta_step': 7,
    'tree_method': 'hist', 'random_state': 42
}

xgb_model = xgb.train(
    params,
    dtrain,
    num_boost_round=100000,
    evals=[(dval, "validation")],
    early_stopping_rounds=50,
    verbose_eval=200
)

y_pred_proba_val = xgb_model.predict(dval)

thresholds = np.arange(0.1, 0.9, 0.02)
f1_scores = [f1_score(y_val_3, (y_pred_proba_val > t).astype(int)) for t in thresholds]

best_threshold = thresholds[np.argmax(f1_scores)]
best_f1 = max(f1_scores)

print(f"\n✅ Meilleur seuil trouvé : {best_threshold:.3f} avec un F1-score de {best_f1:.4f}")

y_pred_proba_test = xgb_model.predict(dtest)
y_pred_test = (y_pred_proba_test > best_threshold).astype(int)

accuracy = accuracy_score(y_test_3, y_pred_test)
precision = precision_score(y_test_3, y_pred_test)
recall = recall_score(y_test_3, y_pred_test)
f1 = f1_score(y_test_3, y_pred_test)

print("\n🔹 **Performance du modèle XGBoost sur le Test Set avec threshold optimisé :**")
print(f"✅ Accuracy:  {accuracy:.4f}")
print(f"✅ Precision: {precision:.4f}")
print(f"✅ Recall:    {recall:.4f}")
print(f"✅ F1-Score:  {f1:.4f}")

---
#### 3.5 Techniques avancées

---
##### 3.5.1 Sous échantillonnage ensemble de Train

In [ ]:
def manual_undersample_with_fraction_binary(X_train, y_train, X_test, y_test, fraction=0.5):
    """
    Performs manual undersampling on the majority class by keeping a specified fraction
    and combines the unselected samples with the original test set for a binary classification problem.

    Parameters:
    - X_train (pd.DataFrame or np.array): The feature matrix for training.
    - y_train (pd.Series or np.array): The target variable for training.
    - X_test (pd.DataFrame or np.array): The feature matrix for testing.
    - y_test (pd.Series or np.array): The target variable for testing.
    - fraction (float): The fraction of the majority class to keep (0 < fraction <= 1).

    Returns:
    - X_resampled (pd.DataFrame or np.array): The undersampled feature matrix for training.
    - y_resampled (pd.Series or np.array): The undersampled target variable for training.
    - X_test_updated (pd.DataFrame or np.array): The updated feature matrix for testing.
    - y_test_updated (pd.Series or np.array): The updated target variable for testing.
    """

    # Combine X_train and y_train into a single DataFrame
    df = pd.concat([X_train, y_train], axis=1)

    # Get the class distribution
    class_counts = y_train.value_counts()
    majority_class = class_counts.idxmax()
    minority_class = class_counts.idxmin()

    # Separate majority and minority class
    df_majority = df[df[y_train.name] == majority_class]
    df_minority = df[df[y_train.name] == minority_class]

    # Undersample the majority class
    num_samples_to_keep = int(len(df_majority) * fraction)
    df_majority_undersampled = df_majority.sample(num_samples_to_keep, random_state=42)

    # Get the unselected samples from the majority class
    unselected_indices = df_majority.index.difference(df_majority_undersampled.index)
    df_unselected = df_majority.loc[unselected_indices]

    # Combine the undersampled majority class with all minority class samples
    df_resampled = pd.concat([df_majority_undersampled, df_minority])
    df_resampled = df_resampled.sample(frac=1, random_state=42).reset_index(drop=True)  # Shuffle

    # Split back into X and y
    X_resampled = df_resampled.drop(columns=y_train.name)
    y_resampled = df_resampled[y_train.name]

    # Combine the unselected samples with the original test set
    X_test_updated = pd.concat([X_test, df_unselected.drop(columns=y_train.name)], axis=0).reset_index(drop=True)
    y_test_updated = pd.concat([y_test, df_unselected[y_train.name]], axis=0).reset_index(drop=True)

    return X_resampled, y_resampled, X_test_updated, y_test_updated

X_train_resampled, y_train_resampled, X_test_updated, y_test_updated = manual_undersample_with_fraction_binary(
    X_train, y_train, X_test, y_test, fraction=0.4
)

print("Répartition des classes après sous-échantillonnage :")
print(y_train_resampled.value_counts())

print("Répartition des classes dans le test mis à jour :")
print(y_test_updated.value_counts())

In [ ]:
for model_name, model in models.items():
    print(f"\n🔹 Test du modèle {model_name} :")

    model.fit(X_train_resampled, y_train_resampled)

    y_pred = model.predict(X_test_updated)

    accuracy = accuracy_score(y_test_updated, y_pred)
    precision = precision_score(y_test_updated, y_pred)
    recall = recall_score(y_test_updated, y_pred)
    f1 = f1_score(y_test_updated, y_pred)

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")

    conf_matrix = confusion_matrix(y_test_updated, y_pred)
    print("Matrice de confusion:")
    print(conf_matrix)
    print('-' * 30)

In [ ]:
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score, accuracy_score
import numpy as np

scoring = {
    'accuracy': make_scorer(accuracy_score),
    'precision': make_scorer(precision_score),
    'recall': make_scorer(recall_score),
    'f1': make_scorer(f1_score)
}

cv_results = {}

for model_name, model in models.items():
    scores = cross_validate(model, X_train_resampled, y_train_resampled, cv=5, scoring=scoring)
    cv_results[model_name] = {
        'accuracy_mean': np.mean(scores['test_accuracy']),
        'accuracy_median': np.median(scores['test_accuracy']),
        'accuracy_std': np.std(scores['test_accuracy']),

        'precision_mean': np.mean(scores['test_precision']),
        'precision_median': np.median(scores['test_precision']),
        'precision_std': np.std(scores['test_precision']),

        'recall_mean': np.mean(scores['test_recall']),
        'recall_median': np.median(scores['test_recall']),
        'recall_std': np.std(scores['test_recall']),

        'f1_mean': np.mean(scores['test_f1']),
        'f1_median': np.median(scores['test_f1']),
        'f1_std': np.std(scores['test_f1']),
    }

for model_name, result in cv_results.items():
    print(f"\n🔹 Modèle: {model_name}")
    print(f"Accuracy: {result['accuracy_mean']:.4f} ± {result['accuracy_std']:.4f} (médiane: {result['accuracy_median']:.4f})")
    print(f"Precision: {result['precision_mean']:.4f} ± {result['precision_std']:.4f} (médiane: {result['precision_median']:.4f})")
    print(f"Recall: {result['recall_mean']:.4f} ± {result['recall_std']:.4f} (médiane: {result['recall_median']:.4f})")
    print(f"F1-Score: {result['f1_mean']:.4f} ± {result['f1_std']:.4f} (médiane: {result['f1_median']:.4f})")
    print('-' * 40)

---
##### 3.5.2 Techniques d'ensemble (Stacking et Réseau de neurones)

---
Stacking

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import StackingClassifier, RandomForestClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import cross_val_score, StratifiedKFold

smote = SMOTE(sampling_strategy='auto', random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

base_models = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')),
    ('dt', DecisionTreeClassifier(random_state=42, class_weight='balanced')),
    ('et', ExtraTreesClassifier(n_estimators=100, random_state=42, class_weight='balanced')),
    ('lgbm', LGBMClassifier(random_state=42, class_weight='balanced')),
    ('xgb', XGBClassifier(random_state=42, scale_pos_weight=int(sum(y_train == 0) / sum(y_train == 1))))
]

meta_model = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42, scale_pos_weight=int(sum(y_train == 0) / sum(y_train == 1)))
stacking_clf = StackingClassifier(estimators=base_models, final_estimator=meta_model, cv=5)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cross_val_results = cross_val_score(stacking_clf, X_train_resampled, y_train_resampled, cv=cv, scoring='accuracy')

#print("📊 Résultats de Cross-Validation :")
#print(f"Mean Accuracy: {cross_val_results.mean():.4f}")
#print(f"Standard Deviation: {cross_val_results.std():.4f}")

stacking_clf.fit(X_train_resampled, y_train_resampled)
y_pred_stacking = stacking_clf.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred_stacking)
precision = precision_score(y_test, y_pred_stacking)
recall = recall_score(y_test, y_pred_stacking)
f1 = f1_score(y_test, y_pred_stacking)

print("\n🔹 **Performance du modèle Stacking :**")
print(f"✅ Accuracy: {accuracy:.4f}")
print(f"✅ Precision: {precision:.4f}")
print(f"✅ Recall: {recall:.4f}")
print(f"✅ F1-Score: {f1:.4f}")

best_precision = 0.5268
best_recall = 0.8750
best_threshold_precision = 0.85
best_threshold_recall = 0.1

y_probs = stacking_clf.predict_proba(X_test_scaled)[:, 1]

for threshold in np.arange(0.1, 0.9, 0.05):
    y_pred_adjusted = (y_probs >= threshold).astype(int)
    precision = precision_score(y_test, y_pred_adjusted)
    recall = recall_score(y_test, y_pred_adjusted)
    if precision > best_precision:
        best_precision = precision
        best_threshold_precision = threshold

    if recall > best_recall:
        best_recall = recall
        best_threshold_recall = threshold

print(f"✅ Meilleur seuil pour la précision : {best_threshold_precision:.2f} avec Précision = {best_precision:.4f}")
print(f"✅ Meilleur seuil pour le rappel : {best_threshold_recall:.2f} avec Rappel = {best_recall:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

y_probs = stacking_clf.predict_proba(X_test_scaled)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_test, y_probs)

plt.figure(figsize=(8, 6))
plt.plot(recalls, precisions, marker='.', label="Stacking Classifier")
plt.xlabel("Rappel")
plt.ylabel("Précision")
plt.title("Courbe Précision-Rappel")
plt.legend()
plt.grid(True)
plt.show()

best_threshold_precision = thresholds[np.argmax(precisions)]
best_threshold_recall = thresholds[np.argmax(recalls)]

---
##### 3.5.3 Expected Calibration Error (ECE)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_resampled, y_train_resampled)

y_probs = model.predict_proba(X_test_scaled)[:, 1]

def expected_calibration_error(y_true, y_probs, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    bin_indices = np.digitize(y_probs, bins) - 1

    ece = 0
    for i in range(n_bins):
        bin_mask = bin_indices == i
        if np.sum(bin_mask) > 0:
            accuracy = np.mean(y_true[bin_mask])
            confidence = np.mean(y_probs[bin_mask])
            ece += (np.abs(accuracy - confidence) * np.sum(bin_mask)) / len(y_probs)

    return ece

from sklearn.calibration import CalibratedClassifierCV

calibrated_model = CalibratedClassifierCV(model, method='isotonic', cv=5)
calibrated_model.fit(X_train_resampled, y_train_resampled)

y_probs_calibrated = calibrated_model.predict_proba(X_test_scaled)[:, 1]

ece_calibrated = expected_calibration_error(y_test.values, y_probs_calibrated)
print(f"ECE après calibration : {ece_calibrated:.4f}")

prob_true_calibrated, prob_pred_calibrated = calibration_curve(y_test, y_probs_calibrated, n_bins=10)

plt.figure(figsize=(6, 6))
plt.plot(prob_pred_calibrated, prob_true_calibrated, marker='o', label="Calibration Curve (Après)")
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label="Parfaite Calibration")
plt.xlabel("Probabilité Moyenne Prédite")
plt.ylabel("Fréquence Réelle")
plt.title("Reliability Diagram (Après Calibration)")
plt.legend()
plt.grid()
plt.show()